# Week 3: Predicting Numbers: Your First Model

**Grade band:** 6 to 8  |  **Duration:** 60 minutes  |  **Platform:** JupyterLite (browser, no account) or Google Colab

**How to use this notebook:** run each cell from top to bottom with Shift + Enter. Read the text, run the code, then complete the challenge cells marked **SOLUTION**. Save your work at the end of the session (File > Download) so it can be uploaded to your portfolio.

> **Teacher copy.** This notebook contains completed answers for every tier. Do not distribute to students. Use it to check work against the validation checklist.

## Hook: The ice cream stand

It will be 92 degrees on Saturday. How many cones should the stand prepare? Guess a number and write it down. By the end of class, a model will make the same prediction from data.

In [ ]:
# Setup: run this cell first.
import pandas as pd
import matplotlib.pyplot as plt

# If a data file is not found next to this notebook (for example on Google Colab),
# it is loaded from the Wize data folder online instead. Replace this URL after publishing.
DATA_URL = "https://raw.githubusercontent.com/wizeacademy/ml-ai-6-8/main/notebooks/data/"

def load(name):
    """Load a Wize dataset by file name, from the local data folder or from the web."""
    try:
        return pd.read_csv("data/" + name)
    except Exception:
        return pd.read_csv(DATA_URL + name)

print("Setup complete. pandas and matplotlib are ready.")

In [ ]:
sales = load("ice_cream_sales.csv")
print(sales.head())

plt.scatter(sales["temperature_f"], sales["cones_sold"], color="#f7941d")
plt.title("Temperature vs cones sold")
plt.xlabel("Temperature (F)")
plt.ylabel("Cones sold")
plt.show()

## Teach 1: What a model is

A **model** is a pattern the computer learned from data that it can use to make predictions. **Regression** predicts a number (cones sold). **Classification** predicts a category (cat or dog). Today is regression.

The scikit-learn recipe is always the same three steps:

1. `model = SomeModel()` create it
2. `model.fit(X, y)` train it on features `X` and labels `y`
3. `model.predict(new_X)` use it

In [ ]:
from sklearn.linear_model import LinearRegression

X = sales[["temperature_f"]]     # features: a table with one column (note the double brackets)
y = sales["cones_sold"]          # label: the number we want to predict

model = LinearRegression()
model.fit(X, y)

print("Every extra degree adds about", round(model.coef_[0], 1), "cones")
prediction = model.predict(pd.DataFrame({"temperature_f": [92]}))
print("Predicted cones at 92 F:", int(prediction[0]))

## Teach 2: Seeing the model

A linear regression model is a straight line drawn through the points. The line is the model. **Concept checkpoint:** if the line goes up to the right, what happens to sales as temperature rises?

In [ ]:
line_x = pd.DataFrame({"temperature_f": [50, 105]})
line_y = model.predict(line_x)

plt.scatter(sales["temperature_f"], sales["cones_sold"], color="#f7941d", label="real days")
plt.plot(line_x["temperature_f"], line_y, color="#102A54", linewidth=3, label="model")
plt.title("The model is the line")
plt.xlabel("Temperature (F)")
plt.ylabel("Cones sold")
plt.legend()
plt.show()

## Teach 3: How wrong is it?

A model is never perfect. The **error** is the gap between what it predicted and what really happened. **Mean absolute error (MAE)** is the average size of that gap.

In [ ]:
from sklearn.metrics import mean_absolute_error

predictions = model.predict(X)
mae = mean_absolute_error(y, predictions)
print("On average the model is off by", round(mae), "cones")

## SOLUTION: Challenge

**Timer suggestion: 25 minutes.**

### Mild
Predict cones sold for 60, 75, and 100 degrees. Print all three with a sentence.

### Medium
Add a second feature: `is_weekend` (1 for Sat and Sun, else 0). Train a new model on both features. Does the MAE go down?

### Spicy
Plot the **residuals** (real minus predicted) against temperature. Are the errors random, or is there a pattern the model misses? Explain in a markdown cell what extra data would help.

In [ ]:
# MILD: three predictions
temps = pd.DataFrame({"temperature_f": [60, 75, 100]})
preds = model.predict(temps)
for t, p in zip(temps["temperature_f"], preds):
    print(f"At {t} F we expect about {int(p)} cones")

In [ ]:
# MEDIUM: add a weekend feature
sales["is_weekend"] = sales["day"].isin(["Sat", "Sun"]).astype(int)
X2 = sales[["temperature_f", "is_weekend"]]
model2 = LinearRegression()
model2.fit(X2, y)
mae2 = mean_absolute_error(y, model2.predict(X2))
print("One feature MAE:", round(mae), " Two feature MAE:", round(mae2))
print("Weekend bonus learned by the model:", round(model2.coef_[1]), "cones")

In [ ]:
# SPICY: residual plot
residuals = y - predictions
plt.scatter(sales["temperature_f"], residuals, color="#2271b1")
plt.axhline(0, color="#102A54", linewidth=2)
plt.title("Residuals: real minus predicted")
plt.xlabel("Temperature (F)")
plt.ylabel("Error (cones)")
plt.show()
# Weekend days sit above the line. Adding is_weekend (Medium) captures that pattern.

## Extra activities (if you finish early)

- What does the model predict at 0 degrees? Is that sensible? Models can be confidently wrong outside their data.
- Invent five rows of "lemonade" data and train a model on them. Is five rows enough to trust?
- Sketch a real business that would pay for a prediction like this.

## Reflection

- Was your Hook guess for 92 degrees close to the model's prediction?
- Why do we measure error instead of just trusting the model?